# Étape 9 - Contrôles qualité des données

Contrôles sur les jeux **Silver** (`sales`, `products`, `stores`, `dates`, `sales_enriched`) et **Gold**, selon 7 dimensions :
complétude, unicité, validité des types, cohérence des valeurs, volumes, clés candidates, intégrité référentielle.

Chaque contrôle reçoit un statut :

| Statut | Sens |
|---|---|
| **OK** | La règle est respectée |
| **ALERTE** | Anomalie restante, connue et acceptée (documentée, signalée par un indicateur ou mise à l'écart) |
| **KO** | La règle n'est pas respectée : une correction est nécessaire |

Résultats : `outputs/rapport_qualite.json`. Rapport rédigé : `docs/quality/rapport_qualite.md`.

**Organisation du notebook**

| Partie | Rôle |
|---|---|
| 0. Paramètres | Chemins, seuils qualité et période attendue lus dans `config/parametres.json` ; journal ; **lecture** des jeux Silver et Gold ; fonction `ctrl` |
| 1 à 7 | **Contrôles** par dimension : complétude, unicité, clés, validité, cohérence, volumes, intégrité |
| Bilan | **Résultats** : `outputs/rapport_qualite.json`, rapport `docs/quality/rapport_qualite.md` |

## 0. Paramètres, chemins, session Spark et journal

Tous les paramètres et chemins viennent de `config/parametres.json` : aucun chemin n'est écrit en dur dans ce notebook.
Le journal d'exécution est ajouté dans `outputs/logs/<notebook>.log`.

In [1]:
import json
import os
from datetime import datetime

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# ---------- Parametres et chemins (config/parametres.json) ----------
RACINE = os.getcwd()
while not os.path.exists(os.path.join(RACINE, "config", "parametres.json")):
    if os.path.dirname(RACINE) == RACINE:
        raise RuntimeError("config/parametres.json introuvable : lancer le notebook depuis le projet")
    RACINE = os.path.dirname(RACINE)
with open(os.path.join(RACINE, "config", "parametres.json")) as f:
    PARAMS = json.load(f)


def chemin(cle):
    """Chemin absolu d'une entree de PARAMS["chemins"] (relative a la racine, ~ accepte)."""
    return os.path.normpath(os.path.join(RACINE, os.path.expanduser(PARAMS["chemins"][cle])))


LOCAL = "file://"  # sans ce prefixe, Spark irait chercher les fichiers dans HDFS (fs.defaultFS)
SOURCE = chemin("source_tpcxbb")
SILVER = LOCAL + chemin("silver")
GOLD = LOCAL + chemin("gold")
OUTPUTS = chemin("outputs")
DOSSIER_SCHEMAS = chemin("schemas")
# store_sales : echantillon ou fichier complet selon le parametre "echantillon.actif"
FICHIER_VENTES = (chemin("echantillon_ventes") if PARAMS["echantillon"]["actif"]
                  else os.path.join(SOURCE, "store_sales", "store_sales.dat"))
os.makedirs(chemin("logs"), exist_ok=True)

# ---------- Journal d'execution ----------
FICHIER_JOURNAL = os.path.join(chemin("logs"), "controles_qualite.log")


def journal(*message):
    ligne = datetime.now().strftime("%Y-%m-%d %H:%M:%S") + " | controles_qualite | " + " ".join(str(m) for m in message)
    print(ligne)
    with open(FICHIER_JOURNAL, "a") as f:
        f.write(ligne + "\n")


# ---------- Session Spark (parametres "spark") ----------
constructeur = SparkSession.builder.appName("TP3-Controles-Qualite").master(PARAMS["spark"]["master"])
for cle, valeur in PARAMS["spark"]["config"].items():
    constructeur = constructeur.config(cle, valeur)
spark = constructeur.getOrCreate()
spark.sparkContext.setLogLevel(PARAMS["spark"]["niveau_journal"])

journal("debut", "| echantillon" if PARAMS["echantillon"]["actif"] else "| fichier complet",
        "| Spark", spark.version, PARAMS["spark"]["master"])

SEUILS = PARAMS["qualite"]
TOL = SEUILS["tolerance_montant"]
DEBUT, FIN = PARAMS["periode_attendue"]["debut"], PARAMS["periode_attendue"]["fin"]

ventes = spark.read.parquet(SILVER + "/sales").cache()
produits = spark.read.parquet(SILVER + "/products").cache()
magasins = spark.read.parquet(SILVER + "/stores").cache()
dates = spark.read.parquet(SILVER + "/dates").cache()
enrichies = spark.read.parquet(SILVER + "/sales_enriched").cache()
GOLD_TABLES = ["ca_mensuel", "ca_annuel", "ca_marge_par_categorie", "top10_produits",
               "ca_marge_par_magasin", "taux_remise_mensuel"]
gold = {nom: spark.read.parquet(GOLD + "/" + nom) for nom in GOLD_TABLES}

NB_VENTES = ventes.count()
controles = []


def ctrl(dimension, jeu, controle, regle, resultat, statut, action=""):
    """Enregistre un controle. statut : OK, ALERTE ou KO."""
    assert statut in ("OK", "ALERTE", "KO")
    controles.append({"dimension": dimension, "jeu": jeu, "controle": controle, "regle": regle,
                      "resultat": str(resultat), "statut": statut, "action": action})
    print(f"[{statut:<6}] {dimension:<12} {jeu:<15} {controle}: {resultat}")


def ok_si(condition, sinon="KO"):
    return "OK" if condition else sinon


def un_vide(colonnes):
    """Vrai si au moins une des colonnes est vide."""
    cond = F.lit(False)
    for col in colonnes:
        cond = cond | F.col(col).isNull()
    return cond


def pct(n, total):
    return f"{n} ({100 * n / total:.2f} %)" if total else str(n)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 17:31:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-11-18 17:32:04 | controles_qualite | debut | echantillon | Spark 4.3.0 local[2]


## 1. Complétude

- **Champs requis** : non nuls. Ce sont les champs nécessaires aux indicateurs et aux jointures.
- **Champs facultatifs** : on mesure leur taux de remplissage. Des vides sont tolérés (ALERTE si le taux reste faible).

In [2]:
REQUIS = {
    "sales": (ventes, ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_ticket_number", "ss_quantity",
                       "ss_sales_price", "ss_ext_sales_price", "ss_net_paid", "ss_net_profit"]),
    "products": (produits, ["i_item_sk", "i_item_id", "i_category", "i_current_price", "i_product_name"]),
    "stores": (magasins, ["s_store_sk", "s_store_id", "s_store_name", "s_city", "s_state"]),
    "dates": (dates, ["d_date_sk", "d_date", "d_year", "d_moy"]),
    "sales_enriched": (enrichies, ["date_vente", "annee", "mois", "categorie", "nom_magasin",
                                   "montant_net_paye", "marge"]),
}
for jeu, (df, cols) in REQUIS.items():
    nb = df.count()
    for c in cols:
        vides = df.filter(F.col(c).isNull()).count()
        ctrl("Completude", jeu, f"{c} renseigne", "Champ requis non nul", pct(vides, nb) + " vide(s)",
             ok_si(vides == 0), "" if vides == 0 else "Rejeter la ligne (regle R3)")

[OK    ] Completude   sales           ss_sold_date_sk renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales           ss_item_sk renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales           ss_store_sk renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales           ss_ticket_number renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales           ss_quantity renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales           ss_sales_price renseigne: 0 (0.00 %) vide(s)


... (12 outputs left out)

[OK    ] Completude   sales_enriched  date_vente renseigne: 0 (0.00 %) vide(s)
[OK    ] Completude   sales_enriched  annee renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales_enriched  mois renseigne: 0 (0.00 %) vide(s)
[OK    ] Completude   sales_enriched  categorie renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales_enriched  nom_magasin renseigne: 0 (0.00 %) vide(s)
[OK    ] Completude   sales_enriched  montant_net_paye renseigne: 0 (0.00 %) vide(s)


[OK    ] Completude   sales_enriched  marge renseigne: 0 (0.00 %) vide(s)


In [3]:
FACULTATIFS = [("sales", ventes, "ss_customer_sk"), ("sales", ventes, "ss_cdemo_sk"), ("sales", ventes, "ss_hdemo_sk"),
               ("sales", ventes, "ss_addr_sk"), ("sales", ventes, "ss_promo_sk"),
               ("products", produits, "i_brand"), ("products", produits, "i_item_desc"),
               ("stores", magasins, "s_closed_date")]
for jeu, df, c in FACULTATIFS:
    nb = df.count()
    vides = df.filter(F.col(c).isNull()).count()
    if c == "s_closed_date":
        statut, action = "ALERTE", "Valeur source incoherente remplacee par null (R5) : demander les vraies dates au fournisseur"
    elif vides == 0:
        statut, action = "OK", ""
    else:
        statut, action = ("ALERTE", "Tolere (< 1 %) : la colonne ne sert pas aux indicateurs") if vides / nb < SEUILS["taux_vide_max"] \
                         else ("KO", "Taux de vide trop eleve : analyser")
    ctrl("Completude", jeu, f"{c} renseigne", "Champ facultatif : taux de vide < 1 %", pct(vides, nb) + " vide(s)",
         statut, action)

for nom, df in gold.items():
    mesures = [c for c, t in df.dtypes if t.startswith(("decimal", "bigint")) and c not in ("evolution_ca_pct", "ca_annee_precedente")]
    vides = df.filter(un_vide(mesures)).count()
    ctrl("Completude", f"gold.{nom}", "mesures renseignees", "Aucune mesure vide (sauf evolution N-1)",
         f"{vides} ligne(s) incomplete(s)", ok_si(vides == 0))

[OK    ] Completude   sales           ss_customer_sk renseigne: 0 (0.00 %) vide(s)


[ALERTE] Completude   sales           ss_cdemo_sk renseigne: 20 (0.03 %) vide(s)


[ALERTE] Completude   sales           ss_hdemo_sk renseigne: 25 (0.04 %) vide(s)


[ALERTE] Completude   sales           ss_addr_sk renseigne: 75 (0.11 %) vide(s)


[ALERTE] Completude   sales           ss_promo_sk renseigne: 15 (0.02 %) vide(s)


[ALERTE] Completude   products        i_brand renseigne: 6 (0.03 %) vide(s)


... (4 outputs left out)

[OK    ] Completude   gold.ca_marge_par_categorie mesures renseignees: 0 ligne(s) incomplete(s)


[OK    ] Completude   gold.top10_produits mesures renseignees: 0 ligne(s) incomplete(s)


[OK    ] Completude   gold.ca_marge_par_magasin mesures renseignees: 0 ligne(s) incomplete(s)


[OK    ] Completude   gold.taux_remise_mensuel mesures renseignees: 0 ligne(s) incomplete(s)


## 2. Unicité et 6. Clés candidates

- **Clé candidate** : on vérifie qu'elle est **unique** et **non nulle**.
- On vérifie aussi qu'elle est **minimale** : avec une colonne en moins, elle ne doit plus être unique. Sinon, la clé aurait des colonnes inutiles.

In [4]:
def doublons(df, cle):
    return df.count() - df.dropDuplicates(cle).count()


CLES = [
    ("sales", ventes, ["ss_ticket_number", "ss_item_sk", "ss_ligne_ticket"]),
    ("products", produits, ["i_item_sk"]), ("products", produits, ["i_item_id"]),
    ("stores", magasins, ["s_store_sk"]), ("stores", magasins, ["s_store_id"]),
    ("dates", dates, ["d_date_sk"]), ("dates", dates, ["d_date"]),
    ("sales_enriched", enrichies, ["ticket_id", "produit_sk", "ligne_ticket"]),
    ("gold.ca_mensuel", gold["ca_mensuel"], ["annee", "mois"]),
    ("gold.ca_annuel", gold["ca_annuel"], ["annee"]),
    ("gold.ca_marge_par_categorie", gold["ca_marge_par_categorie"], ["categorie_id"]),
    ("gold.top10_produits", gold["top10_produits"], ["produit_sk"]),
    ("gold.ca_marge_par_magasin", gold["ca_marge_par_magasin"], ["magasin_sk"]),
    ("gold.taux_remise_mensuel", gold["taux_remise_mensuel"], ["annee", "mois"]),
]
for jeu, df, cle in CLES:
    d = doublons(df, cle)
    nulls = df.filter(un_vide(cle)).count()
    ctrl("Unicite", jeu, f"cle {'+'.join(cle)} unique", "Aucun doublon sur la cle", f"{d} doublon(s)", ok_si(d == 0))
    ctrl("Cle candidate", jeu, f"cle {'+'.join(cle)} non nulle", "Cle toujours renseignee", f"{nulls} cle(s) nulle(s)",
         ok_si(nulls == 0))

[OK    ] Unicite      sales           cle ss_ticket_number+ss_item_sk+ss_ligne_ticket unique: 0 doublon(s)
[OK    ] Cle candidate sales           cle ss_ticket_number+ss_item_sk+ss_ligne_ticket non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      products        cle i_item_sk unique: 0 doublon(s)
[OK    ] Cle candidate products        cle i_item_sk non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      products        cle i_item_id unique: 0 doublon(s)
[OK    ] Cle candidate products        cle i_item_id non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      stores          cle s_store_sk unique: 0 doublon(s)
[OK    ] Cle candidate stores          cle s_store_sk non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      stores          cle s_store_id unique: 0 doublon(s)
[OK    ] Cle candidate stores          cle s_store_id non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      dates           cle d_date_sk unique: 0 doublon(s)
[OK    ] Cle candidate dates           cle d_date_sk non nulle: 0 cle(s) nulle(s)


... (4 outputs left out)

[OK    ] Unicite      gold.ca_marge_par_categorie cle categorie_id unique: 0 doublon(s)
[OK    ] Cle candidate gold.ca_marge_par_categorie cle categorie_id non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      gold.top10_produits cle produit_sk unique: 0 doublon(s)
[OK    ] Cle candidate gold.top10_produits cle produit_sk non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      gold.ca_marge_par_magasin cle magasin_sk unique: 0 doublon(s)
[OK    ] Cle candidate gold.ca_marge_par_magasin cle magasin_sk non nulle: 0 cle(s) nulle(s)


[OK    ] Unicite      gold.taux_remise_mensuel cle annee+mois unique: 0 doublon(s)
[OK    ] Cle candidate gold.taux_remise_mensuel cle annee+mois non nulle: 0 cle(s) nulle(s)


In [5]:
# Minimalite de la cle des ventes : sans ss_ligne_ticket, la cle n'est plus unique
d = doublons(ventes, ["ss_ticket_number", "ss_item_sk"])
ctrl("Cle candidate", "sales", "ticket+produit seul n'est pas une cle",
     "Une ligne de ticket peut repeter le meme produit", f"{d} ligne(s) repetee(s)", "ALERTE",
     "Conservees et numerotees par ss_ligne_ticket (R6) : verifier avec le metier s'il s'agit de vraies lignes")
d = doublons(ventes, ["ss_ticket_number", "ss_ligne_ticket"])
ctrl("Cle candidate", "sales", "ticket+ligne seul n'est pas une cle", "ss_item_sk est necessaire a la cle",
     f"{d} doublon(s) sans ss_item_sk", ok_si(d > 0))
d = doublons(ventes, list(set(ventes.columns) - {c for c in ventes.columns if c.startswith("_")}))
ctrl("Unicite", "sales", "aucune ligne entierement dupliquee", "Pas de copie exacte", f"{d} copie(s)", ok_si(d == 0))

# Un ticket appartient a un seul magasin et une seule date
t = ventes.groupBy("ss_ticket_number").agg(F.countDistinct("ss_store_sk").alias("m"),
                                           F.countDistinct("ss_sold_date_sk").alias("d"))
n = t.filter("m > 1 or d > 1").count()
ctrl("Coherence", "sales", "1 ticket = 1 magasin et 1 date", "Un ticket ne change pas de magasin ni de jour",
     f"{n} ticket(s) incoherent(s)", ok_si(n == 0))

[ALERTE] Cle candidate sales           ticket+produit seul n'est pas une cle: 406 ligne(s) repetee(s)


[OK    ] Cle candidate sales           ticket+ligne seul n'est pas une cle: 57285 doublon(s) sans ss_item_sk


[OK    ] Unicite      sales           aucune ligne entierement dupliquee: 0 copie(s)


[OK    ] Coherence    sales           1 ticket = 1 magasin et 1 date: 0 ticket(s) incoherent(s)


## 3. Validité des types

On compare le type relu dans chaque fichier Parquet au type attendu (spécification TPCx-BB et règle R1).

In [6]:
TYPES = {
    "sales": (ventes, {"ss_sold_date_sk": "bigint", "ss_item_sk": "bigint", "ss_ticket_number": "bigint",
                       "ss_quantity": "int", "ss_sales_price": "decimal(7,2)", "ss_net_paid": "decimal(7,2)",
                       "ss_net_profit": "decimal(7,2)", "ss_prix_nul": "boolean"}),
    "products": (produits, {"i_item_sk": "bigint", "i_current_price": "decimal(7,2)", "i_rec_start_date": "date",
                            "i_category": "string"}),
    "stores": (magasins, {"s_store_sk": "bigint", "s_zip": "string", "s_street_number": "string",
                          "s_rec_start_date": "date", "s_closed_date": "date", "s_tax_precentage": "decimal(5,2)"}),
    "dates": (dates, {"d_date_sk": "bigint", "d_date": "date", "d_year": "int"}),
    "sales_enriched": (enrichies, {"date_vente": "date", "montant_net_paye": "decimal(7,2)", "marge": "decimal(7,2)",
                                   "quantite": "int", "taux_remise": "decimal(12,4)"}),
}
for jeu, (df, attendus) in TYPES.items():
    reels = dict(df.dtypes)
    ecarts = {c: reels.get(c) for c, t in attendus.items() if reels.get(c) != t}
    ctrl("Validite", jeu, "types attendus", f"{len(attendus)} colonnes temoins au bon type",
         ecarts or "conformes", ok_si(not ecarts))
for nom, df in gold.items():
    reels = dict(df.dtypes)
    montants = [c for c in ("chiffre_affaires", "marge", "remises") if c in reels]
    ecarts = {c: reels[c] for c in montants if not reels[c].startswith("decimal")}
    ctrl("Validite", f"gold.{nom}", "montants en decimal", "Pas de double pour les montants",
         ecarts or f"{len(montants)} montant(s) en decimal", ok_si(not ecarts))

[OK    ] Validite     sales           types attendus: conformes
[OK    ] Validite     products        types attendus: conformes
[OK    ] Validite     stores          types attendus: conformes
[OK    ] Validite     dates           types attendus: conformes
[OK    ] Validite     sales_enriched  types attendus: conformes
[OK    ] Validite     gold.ca_mensuel montants en decimal: 3 montant(s) en decimal
[OK    ] Validite     gold.ca_annuel  montants en decimal: 3 montant(s) en decimal
[OK    ] Validite     gold.ca_marge_par_categorie montants en decimal: 3 montant(s) en decimal
[OK    ] Validite     gold.top10_produits montants en decimal: 3 montant(s) en decimal
[OK    ] Validite     gold.ca_marge_par_magasin montants en decimal: 3 montant(s) en decimal
[OK    ] Validite     gold.taux_remise_mensuel montants en decimal: 1 montant(s) en decimal


In [7]:
# Validite des formats texte
n = magasins.filter(~F.col("s_zip").rlike("^[0-9]{5}$")).count()
ctrl("Validite", "stores", "s_zip = 5 chiffres", "Format ^[0-9]{5}$", f"{n} invalide(s)", ok_si(n == 0))
n = magasins.filter(~F.col("s_state").rlike("^[A-Z]{2}$")).count()
ctrl("Validite", "stores", "s_state = 2 lettres", "Format ^[A-Z]{2}$", f"{n} invalide(s)", ok_si(n == 0))
n = produits.filter(~F.col("i_item_id").rlike("^[A-Z]{16}$")).count()
ctrl("Validite", "products", "i_item_id = 16 lettres", "Format ^[A-Z]{16}$", f"{n} invalide(s)", ok_si(n == 0))
n = magasins.filter(~F.col("s_hours").rlike("^[0-9]{1,2}(AM|PM)-[0-9]{1,2}(AM|PM)$")).count()
ctrl("Validite", "stores", "s_hours au format hAM-hPM", "Format ^h(AM|PM)-h(AM|PM)$", f"{n} invalide(s)", ok_si(n == 0))

[OK    ] Validite     stores          s_zip = 5 chiffres: 0 invalide(s)
[OK    ] Validite     stores          s_state = 2 lettres: 0 invalide(s)


[OK    ] Validite     products        i_item_id = 16 lettres: 0 invalide(s)
[OK    ] Validite     stores          s_hours au format hAM-hPM: 0 invalide(s)


## 4. Cohérence des valeurs

- **Domaines de valeurs** : les valeurs doivent rester dans un domaine valide (bornes, listes connues).
- **Relations entre colonnes** : on vérifie les formules TPCx-BB, par exemple montant net = montant de vente − coupon.
- **Anomalies métier conservées** : elles sont signalées en ALERTE.

In [8]:
def regle(jeu, df, nom, regle_txt, condition_invalide, statut_si_invalide="KO", action=""):
    n = df.filter(condition_invalide).count()
    total = df.count()
    ctrl("Coherence", jeu, nom, regle_txt, pct(n, total) + " hors regle",
         "OK" if n == 0 else statut_si_invalide, "" if n == 0 else action)


c = F.col
regle("sales", ventes, "quantite dans [1, 100]", "1 <= ss_quantity <= 100", (c("ss_quantity") < 1) | (c("ss_quantity") > 100))
regle("sales", ventes, "prix >= 0", "Prix et montants payes positifs ou nuls",
      (c("ss_sales_price") < 0) | (c("ss_list_price") < 0) | (c("ss_net_paid") < 0))
regle("sales", ventes, "prix vente <= prix catalogue", "ss_sales_price <= ss_list_price", c("ss_sales_price") > c("ss_list_price"))
regle("sales", ventes, "montant = quantite x prix", "ss_ext_sales_price = ss_quantity x ss_sales_price",
      F.abs(c("ss_ext_sales_price") - c("ss_quantity") * c("ss_sales_price")) > TOL)
regle("sales", ventes, "montant catalogue = quantite x prix catalogue", "ss_ext_list_price = ss_quantity x ss_list_price",
      F.abs(c("ss_ext_list_price") - c("ss_quantity") * c("ss_list_price")) > TOL)
regle("sales", ventes, "remise = catalogue - vente", "ss_ext_discount_amt = ss_ext_list_price - ss_ext_sales_price",
      F.abs(c("ss_ext_discount_amt") - (c("ss_ext_list_price") - c("ss_ext_sales_price"))) > TOL, "ALERTE",
      "Formule de remise differente de celle supposee : a documenter dans le dictionnaire de KPI")
regle("sales", ventes, "net paye = vente - coupon", "ss_net_paid = ss_ext_sales_price - ss_coupon_amt",
      F.abs(c("ss_net_paid") - (c("ss_ext_sales_price") - c("ss_coupon_amt"))) > TOL)
regle("sales", ventes, "net TTC = net + taxe", "ss_net_paid_inc_tax = ss_net_paid + ss_ext_tax",
      F.abs(c("ss_net_paid_inc_tax") - (c("ss_net_paid") + c("ss_ext_tax"))) > TOL)
regle("sales", ventes, "marge = net paye - cout", "ss_net_profit = ss_net_paid - ss_ext_wholesale_cost",
      F.abs(c("ss_net_profit") - (c("ss_net_paid") - c("ss_ext_wholesale_cost"))) > TOL)
regle("sales", ventes, "coupon <= montant de vente", "ss_coupon_amt <= ss_ext_sales_price",
      c("ss_coupon_amt") > c("ss_ext_sales_price"))

[OK    ] Coherence    sales           quantite dans [1, 100]: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           prix >= 0: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           prix vente <= prix catalogue: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           montant = quantite x prix: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           montant catalogue = quantite x prix catalogue: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           remise = catalogue - vente: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           net paye = vente - coupon: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           net TTC = net + taxe: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           marge = net paye - cout: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           coupon <= montant de vente: 0 (0.00 %) hors regle


In [9]:
# Anomalies metier conservees (ALERTE) : ventes reelles mais a interpreter avec prudence
regle("sales", ventes, "marge positive", "ss_net_profit >= 0", c("ss_net_profit") < 0, "ALERTE",
      "Conservee (R7), indicateur ss_marge_negative : donnees generees, marge a comparer et non a juger")
regle("sales", ventes, "vente au-dessus du cout", "ss_sales_price >= ss_wholesale_cost",
      c("ss_sales_price") < c("ss_wholesale_cost"), "ALERTE", "Conservee : vente a perte, cas metier possible")
regle("sales", ventes, "prix de vente non nul", "ss_sales_price > 0", c("ss_sales_price") == 0, "ALERTE",
      "Conservee (R7), indicateur ss_prix_nul : article offert ? a confirmer avec le metier")
regle("products", produits, "cout d'achat <= prix courant", "i_wholesale_cost <= i_current_price",
      c("i_wholesale_cost") > c("i_current_price"), "ALERTE", "Conservee, indicateur i_cout_superieur_prix")
regle("stores", magasins, "horaires d'ouverture coherents", "Heure d'ouverture differente de l'heure de fermeture",
      F.split("s_hours", "-")[0] == F.split("s_hours", "-")[1], "ALERTE",
      "Conservee (8AM-8AM : ouvert 24 h/24 ?), colonne non utilisee par les indicateurs")
regle("stores", magasins, "date de fermeture fiable", "s_date_fermeture_invalide = false",
      c("s_date_fermeture_invalide"), "ALERTE", "Date remplacee par null (R5)")

[ALERTE] Coherence    sales           marge positive: 49559 (74.42 %) hors regle


[ALERTE] Coherence    sales           vente au-dessus du cout: 46271 (69.48 %) hors regle


[ALERTE] Coherence    sales           prix de vente non nul: 356 (0.53 %) hors regle


[ALERTE] Coherence    products        cout d'achat <= prix courant: 7768 (43.59 %) hors regle


[ALERTE] Coherence    stores          horaires d'ouverture coherents: 1 (8.33 %) hors regle


[ALERTE] Coherence    stores          date de fermeture fiable: 12 (100.00 %) hors regle


In [10]:
# Domaines de valeurs
CATEGORIES = ["Books", "Clothing & Accessories", "Electronics", "Home & Kitchen", "Movies & TV", "Music",
              "Sports & Outdoors", "Tools & Home Improvement", "Toys & Games"]
regle("products", produits, "categorie dans le referentiel", "i_category parmi les 9 categories TPCx-BB",
      ~c("i_category").isin(CATEGORIES))
n = produits.groupBy("i_category_id").agg(F.countDistinct("i_category").alias("n")).filter("n > 1").count()
ctrl("Coherence", "products", "1 code = 1 categorie", "i_category_id -> un seul libelle", f"{n} code(s) ambigu(s)", ok_si(n == 0))
regle("dates", dates, "mois dans [1, 12]", "1 <= d_moy <= 12", (c("d_moy") < 1) | (c("d_moy") > 12))
regle("dates", dates, "annee = annee de d_date", "d_year = year(d_date)", c("d_year") != F.year("d_date"))
regle("sales_enriched", enrichies, "date de vente dans la periode", f"{DEBUT} <= date_vente <= {FIN}",
      (c("date_vente") < DEBUT) | (c("date_vente") > FIN))
regle("sales_enriched", enrichies, "taux de remise dans [0, 1]", "0 <= taux_remise <= 1",
      (c("taux_remise") < 0) | (c("taux_remise") > 1))
vente_avant_ouverture = (ventes.join(dates, ventes.ss_sold_date_sk == dates.d_date_sk)
                         .join(magasins, ventes.ss_store_sk == magasins.s_store_sk)
                         .filter(c("d_date") < c("s_rec_start_date")).count())
ctrl("Coherence", "sales", "vente apres ouverture du magasin", "d_date >= s_rec_start_date",
     f"{vente_avant_ouverture} vente(s) avant ouverture", ok_si(vente_avant_ouverture == 0))

[OK    ] Coherence    products        categorie dans le referentiel: 0 (0.00 %) hors regle


[OK    ] Coherence    products        1 code = 1 categorie: 0 code(s) ambigu(s)


[OK    ] Coherence    dates           mois dans [1, 12]: 0 (0.00 %) hors regle


[OK    ] Coherence    dates           annee = annee de d_date: 0 (0.00 %) hors regle


[OK    ] Coherence    sales_enriched  date de vente dans la periode: 0 (0.00 %) hors regle


[OK    ] Coherence    sales_enriched  taux de remise dans [0, 1]: 0 (0.00 %) hors regle


[OK    ] Coherence    sales           vente apres ouverture du magasin: 0 vente(s) avant ouverture


In [11]:
# Coherence Gold <-> Silver : les totaux doivent se retrouver
ca_silver = enrichies.agg(F.sum("montant_net_paye")).first()[0]
marge_silver = enrichies.agg(F.sum("marge")).first()[0]
for nom in ["ca_mensuel", "ca_annuel", "ca_marge_par_categorie", "ca_marge_par_magasin"]:
    s = gold[nom].agg(F.sum("chiffre_affaires"), F.sum("marge")).first()
    ctrl("Coherence", f"gold.{nom}", "totaux = Silver", "Somme CA et marge = sales_enriched",
         f"CA {s[0]} / {ca_silver}, marge {s[1]} / {marge_silver}", ok_si(s[0] == ca_silver and s[1] == marge_silver))
n = gold["taux_remise_mensuel"].filter((c("taux_remise_pct") < 0) | (c("taux_remise_pct") > 100)).count()
ctrl("Coherence", "gold.taux_remise_mensuel", "taux dans [0, 100]", "0 <= taux_remise_pct <= 100", f"{n} hors bornes", ok_si(n == 0))
parts = gold["ca_marge_par_categorie"].agg(F.sum("part_ca_pct")).first()[0]
ctrl("Coherence", "gold.ca_marge_par_categorie", "somme des parts = 100 %", "Σ part_ca_pct = 100 (± 0,1 arrondi)",
     f"{parts} %", ok_si(abs(float(parts) - 100) <= 0.1))
top = gold["top10_produits"].agg(F.min("chiffre_affaires")).first()[0]
mieux_hors_top = (enrichies.groupBy("produit_sk").agg(F.sum("montant_net_paye").alias("ca"))
                  .join(gold["top10_produits"].select("produit_sk"), "produit_sk", "left_anti")
                  .filter(c("ca") > top).count())
ctrl("Coherence", "gold.top10_produits", "top 10 exact", "Aucun produit hors top 10 avec un CA superieur",
     f"{mieux_hors_top} produit(s)", ok_si(mieux_hors_top == 0))
n = gold["ca_mensuel"].filter("mois_incomplet").count()
ctrl("Coherence", "gold.ca_mensuel", "mois complets", "Chaque mois a des donnees jusqu'a son dernier jour",
     f"{n} mois incomplet(s) (decembre 2005)", "ALERTE" if n else "OK",
     "Signale par mois_incomplet : ne pas comparer decembre 2005 a decembre 2004")

[OK    ] Coherence    gold.ca_mensuel totaux = Silver: CA 115752176.68 / 115752176.68, marge -55241252.10 / -55241252.10
[OK    ] Coherence    gold.ca_annuel  totaux = Silver: CA 115752176.68 / 115752176.68, marge -55241252.10 / -55241252.10


[OK    ] Coherence    gold.ca_marge_par_categorie totaux = Silver: CA 115752176.68 / 115752176.68, marge -55241252.10 / -55241252.10
[OK    ] Coherence    gold.ca_marge_par_magasin totaux = Silver: CA 115752176.68 / 115752176.68, marge -55241252.10 / -55241252.10


[OK    ] Coherence    gold.taux_remise_mensuel taux dans [0, 100]: 0 hors bornes


[OK    ] Coherence    gold.ca_marge_par_categorie somme des parts = 100 %: 100.01 %


[OK    ] Coherence    gold.top10_produits top 10 exact: 0 produit(s)
[ALERTE] Coherence    gold.ca_mensuel mois complets: 1 mois incomplet(s) (decembre 2005)


## 5. Volumes

In [12]:
with open(os.path.join(OUTPUTS, "bilan_nettoyage.json")) as f:
    bilan = json.load(f)

for jeu, table, df in [("sales", "store_sales", ventes), ("products", "item", produits),
                       ("stores", "store", magasins), ("dates", "date_dim", dates)]:
    lus, nb = bilan[table]["lignes_lues"], df.count()
    rejets = spark.read.parquet(SILVER + "/rejects/" + jeu).count()
    ctrl("Volume", jeu, "lignes lues = Silver + rejets", "Aucune ligne perdue sans trace",
         f"{lus} lues = {nb} + {rejets} rejet(s)", ok_si(lus == nb + rejets))
    ctrl("Volume", jeu, "donnees mises a l'ecart", "Lignes rejetees, conservees dans rejects/",
         f"{rejets} ligne(s)", "OK")

ctrl("Volume", "sales_enriched", "lignes = sales", "Jointures N:1 sans perte ni duplication",
     f"{enrichies.count()} / {NB_VENTES}", ok_si(enrichies.count() == NB_VENTES))

# Echantillon : environ 10 % des tickets du fichier complet
complet = spark.read.option("sep", "|").csv(LOCAL + os.path.join(SOURCE, "store_sales", "store_sales.dat")) \
              .select(F.col("_c9").alias("ticket"))
nb_complet, tickets_complet = complet.count(), complet.distinct().count()
tickets = ventes.select("ss_ticket_number").distinct().count()
ctrl("Volume", "sales", "echantillon = 10 % des tickets", "Tickets echantillon / tickets complets ~ 10 %",
     f"{tickets} / {tickets_complet} tickets ({100 * tickets / tickets_complet:.2f} %), "
     f"{NB_VENTES} / {nb_complet} lignes ({100 * NB_VENTES / nb_complet:.2f} %)",
     "ALERTE", "Limite connue : passer au fichier complet sur une machine plus puissante")

[OK    ] Volume       sales           lignes lues = Silver + rejets: 66593 lues = 66593 + 0 rejet(s)
[OK    ] Volume       sales           donnees mises a l'ecart: 0 ligne(s)


[OK    ] Volume       products        lignes lues = Silver + rejets: 17820 lues = 17820 + 0 rejet(s)
[OK    ] Volume       products        donnees mises a l'ecart: 0 ligne(s)


[OK    ] Volume       stores          lignes lues = Silver + rejets: 12 lues = 12 + 0 rejet(s)
[OK    ] Volume       stores          donnees mises a l'ecart: 0 ligne(s)


[OK    ] Volume       dates           lignes lues = Silver + rejets: 109573 lues = 109573 + 0 rejet(s)
[OK    ] Volume       dates           donnees mises a l'ecart: 0 ligne(s)
[OK    ] Volume       sales_enriched  lignes = sales: 66593 / 66593


[ALERTE] Volume       sales           echantillon = 10 % des tickets: 8910 / 89100 tickets (10.00 %), 66593 / 667579 lignes (9.98 %)


In [13]:
# Repartition : chaque mois et chaque magasin ont des ventes, sans trou ni pic anormal
par_mois = enrichies.groupBy("annee", "mois").count()
nb_mois = par_mois.count()
stats = par_mois.agg(F.min("count"), F.max("count"), F.avg("count")).first()
ctrl("Volume", "sales_enriched", "60 mois presents", "Un mois sans vente serait un trou de chargement",
     f"{nb_mois} mois, de {stats[0]} a {stats[1]} lignes (moyenne {stats[2]:.0f})", ok_si(nb_mois == 60))
faibles = par_mois.filter(c("count") < SEUILS["ratio_mois_min"] * stats[2]).collect()
ctrl("Volume", "sales_enriched", "pas de mois anormalement faible", "Lignes du mois >= 50 % de la moyenne",
     [f"{r['annee']}-{r['mois']:02d} : {r['count']}" for r in faibles] or "aucun",
     "ALERTE" if faibles else "OK", "Decembre 2005 partiel (jusqu'au 13/12)" if faibles else "")
par_magasin = enrichies.groupBy("magasin_sk").count()
s = par_magasin.agg(F.count("*"), F.min("count"), F.max("count")).first()
ctrl("Volume", "sales_enriched", "12 magasins avec des ventes", "Chaque magasin a des ventes",
     f"{s[0]} magasins, de {s[1]} a {s[2]} lignes", ok_si(s[0] == 12))
for nom, attendu in [("ca_mensuel", 60), ("ca_annuel", 5), ("ca_marge_par_categorie", 9), ("top10_produits", 10),
                     ("ca_marge_par_magasin", 12), ("taux_remise_mensuel", 60)]:
    n = gold[nom].count()
    ctrl("Volume", f"gold.{nom}", f"{attendu} lignes", "Une ligne par valeur de la granularite", n, ok_si(n == attendu))

[OK    ] Volume       sales_enriched  60 mois presents: 60 mois, de 425 a 1271 lignes (moyenne 1110)
[ALERTE] Volume       sales_enriched  pas de mois anormalement faible: ['2005-12 : 425']


[OK    ] Volume       sales_enriched  12 magasins avec des ventes: 12 magasins, de 5172 a 5891 lignes
[OK    ] Volume       gold.ca_mensuel 60 lignes: 60


[OK    ] Volume       gold.ca_annuel  5 lignes: 5
[OK    ] Volume       gold.ca_marge_par_categorie 9 lignes: 9


[OK    ] Volume       gold.top10_produits 10 lignes: 10
[OK    ] Volume       gold.ca_marge_par_magasin 12 lignes: 12
[OK    ] Volume       gold.taux_remise_mensuel 60 lignes: 60


## 7. Intégrité référentielle

- Contre les **référentiels Silver** : produits, magasins, dates.
- Contre des **tables brutes non retenues** (`promotion`, `customer`, `time_dim`), pour vérifier les clés que nous gardons sans les utiliser. On ne lit que la colonne de clé, pour rester léger.

In [14]:
def orphelines(df, cle, ref, cle_ref):
    return df.filter(F.col(cle).isNotNull()).join(ref, df[cle] == ref[cle_ref], "left_anti").count()


def cle_brute(table, indice):
    return (spark.read.option("sep", "|").csv(LOCAL + os.path.join(SOURCE, table, f"{table}.dat"))
                 .select(F.col(f"_c{indice}").cast("bigint").alias("cle")).distinct())


RELATIONS = [
    ("sales", "ss_item_sk", "silver/products", produits, "i_item_sk"),
    ("sales", "ss_store_sk", "silver/stores", magasins, "s_store_sk"),
    ("sales", "ss_sold_date_sk", "silver/dates", dates, "d_date_sk"),
    ("sales", "ss_promo_sk", "brut/promotion", cle_brute("promotion", 0), "cle"),
    ("sales", "ss_customer_sk", "brut/customer", cle_brute("customer", 0), "cle"),
    ("sales", "ss_sold_time_sk", "brut/time_dim", cle_brute("time_dim", 0), "cle"),
    ("sales_enriched", "produit_sk", "silver/products", produits, "i_item_sk"),
    ("sales_enriched", "magasin_sk", "silver/stores", magasins, "s_store_sk"),
]
for jeu, cle, nom_ref, ref, cle_ref in RELATIONS:
    df = ventes if jeu == "sales" else enrichies
    n = orphelines(df, cle, ref, cle_ref)
    ctrl("Integrite", jeu, f"{cle} -> {nom_ref}", "Chaque cle renseignee existe dans la reference",
         f"{n} orpheline(s)", ok_si(n == 0), "" if n == 0 else "Rejeter ou completer la reference (R4)")

for nom, cle, ref, cle_ref in [("ca_marge_par_categorie", "categorie_id", produits, "i_category_id"),
                               ("ca_marge_par_magasin", "magasin_sk", magasins, "s_store_sk"),
                               ("top10_produits", "produit_sk", produits, "i_item_sk")]:
    n = orphelines(gold[nom], cle, ref.select(cle_ref).distinct(), cle_ref)
    ctrl("Integrite", f"gold.{nom}", f"{cle} -> Silver", "Chaque dimension Gold existe en Silver",
         f"{n} orpheline(s)", ok_si(n == 0))

[OK    ] Integrite    sales           ss_item_sk -> silver/products: 0 orpheline(s)


[OK    ] Integrite    sales           ss_store_sk -> silver/stores: 0 orpheline(s)


[OK    ] Integrite    sales           ss_sold_date_sk -> silver/dates: 0 orpheline(s)


[OK    ] Integrite    sales           ss_promo_sk -> brut/promotion: 0 orpheline(s)


[OK    ] Integrite    sales           ss_customer_sk -> brut/customer: 0 orpheline(s)


[OK    ] Integrite    sales           ss_sold_time_sk -> brut/time_dim: 0 orpheline(s)


[OK    ] Integrite    sales_enriched  produit_sk -> silver/products: 0 orpheline(s)
[OK    ] Integrite    sales_enriched  magasin_sk -> silver/stores: 0 orpheline(s)


[OK    ] Integrite    gold.ca_marge_par_categorie categorie_id -> Silver: 0 orpheline(s)


[OK    ] Integrite    gold.ca_marge_par_magasin magasin_sk -> Silver: 0 orpheline(s)


[OK    ] Integrite    gold.top10_produits produit_sk -> Silver: 0 orpheline(s)


## Bilan

In [15]:
from collections import Counter

par_statut = Counter(r["statut"] for r in controles)
print(f"{len(controles)} controles : {dict(par_statut)}")
print()
print("Par dimension :")
for dim in ["Completude", "Unicite", "Validite", "Coherence", "Volume", "Cle candidate", "Integrite"]:
    s = Counter(r["statut"] for r in controles if r["dimension"] == dim)
    print(f"  {dim:<14} {sum(s.values()):>3} controle(s) | OK {s['OK']:>3} | ALERTE {s['ALERTE']:>2} | KO {s['KO']:>2}")
print()
print("Anomalies restantes (ALERTE / KO) :")
for r in controles:
    if r["statut"] != "OK":
        print(f"  [{r['statut']}] {r['jeu']} - {r['controle']} : {r['resultat']}\n           -> {r['action']}")

with open(os.path.join(OUTPUTS, "rapport_qualite.json"), "w") as f:
    json.dump({"synthese": dict(par_statut), "controles": controles}, f, indent=1, ensure_ascii=False)

152 controles : {'OK': 135, 'ALERTE': 17}

Par dimension :
  Completude      44 controle(s) | OK  37 | ALERTE  7 | KO  0
  Unicite         15 controle(s) | OK  15 | ALERTE  0 | KO  0
  Validite        15 controle(s) | OK  15 | ALERTE  0 | KO  0
  Coherence       32 controle(s) | OK  25 | ALERTE  7 | KO  0
  Volume          19 controle(s) | OK  17 | ALERTE  2 | KO  0
  Cle candidate   16 controle(s) | OK  15 | ALERTE  1 | KO  0
  Integrite       11 controle(s) | OK  11 | ALERTE  0 | KO  0

Anomalies restantes (ALERTE / KO) :
  [ALERTE] sales - ss_cdemo_sk renseigne : 20 (0.03 %) vide(s)
           -> Tolere (< 1 %) : la colonne ne sert pas aux indicateurs
  [ALERTE] sales - ss_hdemo_sk renseigne : 25 (0.04 %) vide(s)
           -> Tolere (< 1 %) : la colonne ne sert pas aux indicateurs
  [ALERTE] sales - ss_addr_sk renseigne : 75 (0.11 %) vide(s)
           -> Tolere (< 1 %) : la colonne ne sert pas aux indicateurs
  [ALERTE] sales - ss_promo_sk renseigne : 15 (0.02 %) vide(s)
         

In [16]:
journal("fin | OK")
spark.stop()

2026-11-18 17:33:07 | controles_qualite | fin | OK
